# 第9章：深度学习模型

## 本章学习目标

- 掌握 PyTorch 模型开发流程
- 理解时序模型结构
- 学会模型训练与调优
- 能够对比不同模型效果

---

## 9.1 深度学习模型概述

Qlib 支持多种深度学习模型，主要用于捕捉股票价格的时序特征。

### 模型类型

| 模型 | 特点 | 适用场景 |
|------|------|----------|
| **LSTM** | 长短期记忆网络 | 时序预测 |
| GRU | 门控循环单元 | 轻量级时序 |
| ALSTM | 注意力LSTM | 关注重要时间点 |
| Transformer | 自注意力机制 | 长距离依赖 |
| GATs | 图注意力网络 | 股票关联 |

### 时序建模特点

```
输入: 过去 T 天的特征
       ┌─────────────────────────────────┐
       │ t-T+1  t-T+2  ...  t-1    t     │
       │  ↓      ↓           ↓     ↓    │
       │ [特征] [特征] ... [特征] [特征] │
       └─────────────────────────────────┘
                      ↓
              时序模型 (LSTM/GRU/...)
                      ↓
              预测: t+1 的收益
```

In [ ]:
import qlib
from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import Alpha360
from qlib.workflow import R
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 9.2 数据准备

深度学习模型通常使用 Alpha360 特征集，因为其包含了更长时间窗口的特征。

In [ ]:
# 创建数据集
# 注意：深度学习模型需要使用 TSDataSampler
# cn_data 数据最晚到 2020-09-25
from qlib.data.dataset import TSDatasetH

dataset = TSDatasetH(
    handler={
        "class": "Alpha360",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2015-01-01",
            "end_time": "2020-09-25",  # 数据最晚日期
            "fit_start_time": "2015-01-01",
            "fit_end_time": "2018-12-31",
            "instruments": "csi300",
        },
    },
    segments={
        "train": ("2015-01-01", "2018-12-31"),
        "valid": ("2019-01-01", "2020-06-30"),
        "test": ("2020-07-01", "2020-09-25"),  # 数据最晚日期
    },
)

print("数据集创建完成")

In [ ]:
# 查看数据形状
train_data = dataset.prepare("train")
print(f"训练数据形状: {train_data.shape}")
train_data.head()

## 9.3 LSTM 模型

In [ ]:
# 导入 LSTM 模型
from qlib.contrib.model.pytorch_lstm import LSTM

# 创建 LSTM 模型
lstm_model = LSTM(
    d_feat=6,           # 输入特征维度 (Alpha360 中每只股票每天有 6 个基础特征)
    hidden_size=64,     # 隐藏层大小
    num_layers=2,       # LSTM 层数
    dropout=0.0,        # Dropout 比例
    n_epochs=50,        # 训练轮数
    lr=0.001,           # 学习率
    early_stop=10,      # 早停轮数
    batch_size=800,     # 批大小
    metric="loss",      # 评估指标
    loss="mse",         # 损失函数
    n_jobs=4,           # 并行数
    GPU=0,              # GPU ID，-1 表示使用 CPU
    seed=42,
)

print("LSTM 模型配置:")
print(f"  hidden_size: {lstm_model.hidden_size}")
print(f"  num_layers: {lstm_model.num_layers}")
print(f"  n_epochs: {lstm_model.n_epochs}")
print(f"  batch_size: {lstm_model.batch_size}")

In [ ]:
# 训练模型
print("开始训练 LSTM 模型...")
lstm_model.fit(dataset)
print("训练完成")

In [ ]:
# 预测
lstm_predictions = lstm_model.predict(dataset)

print(f"预测结果形状: {lstm_predictions.shape}")
lstm_predictions.head()

In [ ]:
# 评估
def evaluate_predictions(predictions, labels):
    """评估预测结果"""
    pred = np.array(predictions).ravel()
    label = np.array(labels).ravel()
    
    mask = ~(np.isnan(pred) | np.isnan(label))
    pred = pred[mask]
    label = label[mask]
    
    ic = np.corrcoef(pred, label)[0, 1]
    rank_ic = np.corrcoef(np.argsort(np.argsort(pred)), np.argsort(np.argsort(label)))[0, 1]
    
    return {"IC": ic, "Rank IC": rank_ic, "样本数": len(pred)}

test_data = dataset.prepare("test")
labels = test_data['label']

lstm_metrics = evaluate_predictions(lstm_predictions, labels)

print("LSTM 模型评估结果:")
for k, v in lstm_metrics.items():
    print(f"  {k}: {v:.4f}")

## 9.4 GRU 模型

In [ ]:
# 导入 GRU 模型
from qlib.contrib.model.pytorch_gru import GRU

# 创建 GRU 模型
gru_model = GRU(
    d_feat=6,
    hidden_size=64,
    num_layers=2,
    dropout=0.0,
    n_epochs=50,
    lr=0.001,
    early_stop=10,
    batch_size=800,
    metric="loss",
    loss="mse",
    n_jobs=4,
    GPU=-1,  # 使用 CPU
    seed=42,
)

print("开始训练 GRU 模型...")
gru_model.fit(dataset)
print("训练完成")

In [ ]:
# 预测并评估
gru_predictions = gru_model.predict(dataset)
gru_metrics = evaluate_predictions(gru_predictions, labels)

print("GRU 模型评估结果:")
for k, v in gru_metrics.items():
    print(f"  {k}: {v:.4f}")

## 9.5 Transformer 模型

In [ ]:
# 导入 Transformer 模型
from qlib.contrib.model.pytorch_transformer import Transformer

# 创建 Transformer 模型
transformer_model = Transformer(
    d_feat=6,
    hidden_size=64,
    num_layers=2,
    dropout=0.0,
    n_epochs=50,
    lr=0.001,
    early_stop=10,
    batch_size=800,
    metric="loss",
    loss="mse",
    n_jobs=4,
    GPU=-1,
    seed=42,
)

print("开始训练 Transformer 模型...")
transformer_model.fit(dataset)
print("训练完成")

In [ ]:
# 预测并评估
transformer_predictions = transformer_model.predict(dataset)
transformer_metrics = evaluate_predictions(transformer_predictions, labels)

print("Transformer 模型评估结果:")
for k, v in transformer_metrics.items():
    print(f"  {k}: {v:.4f}")

## 9.6 模型对比

In [ ]:
# 汇总对比
comparison = pd.DataFrame({
    "LSTM": lstm_metrics,
    "GRU": gru_metrics,
    "Transformer": transformer_metrics,
}).T

print("深度学习模型对比:")
comparison

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# IC 对比
models = ['LSTM', 'GRU', 'Transformer']
ics = [lstm_metrics['IC'], gru_metrics['IC'], transformer_metrics['IC']]
colors = ['steelblue', 'coral', 'green']

axes[0].bar(models, ics, color=colors)
axes[0].set_title('模型 IC 对比')
axes[0].set_ylabel('IC')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Rank IC 对比
rank_ics = [lstm_metrics['Rank IC'], gru_metrics['Rank IC'], transformer_metrics['Rank IC']]

axes[1].bar(models, rank_ics, color=colors)
axes[1].set_title('模型 Rank IC 对比')
axes[1].set_ylabel('Rank IC')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 9.7 使用 Recorder 管理实验

In [ ]:
# 使用 Recorder 记录实验
with R.start(experiment_name="deep_learning_comparison") as recorder:
    
    # 记录 LSTM 结果
    recorder.log_params({"LSTM_hidden_size": 64, "LSTM_num_layers": 2})
    recorder.log_metrics({"LSTM_IC": lstm_metrics['IC'], "LSTM_RankIC": lstm_metrics['Rank IC']})
    
    # 记录 GRU 结果
    recorder.log_params({"GRU_hidden_size": 64, "GRU_num_layers": 2})
    recorder.log_metrics({"GRU_IC": gru_metrics['IC'], "GRU_RankIC": gru_metrics['Rank IC']})
    
    # 记录 Transformer 结果
    recorder.log_params({"Transformer_hidden_size": 64, "Transformer_num_layers": 2})
    recorder.log_metrics({"Transformer_IC": transformer_metrics['IC'], "Transformer_RankIC": transformer_metrics['Rank IC']})
    
    # 保存最佳模型
    best_model_name = max(['LSTM', 'GRU', 'Transformer'], 
                          key=lambda x: {'LSTM': lstm_metrics['IC'], 
                                         'GRU': gru_metrics['IC'], 
                                         'Transformer': transformer_metrics['IC']}[x])
    print(f"最佳模型: {best_model_name}")
    
    print(f"\nRecorder ID: {recorder.id}")

## 9.8 训练技巧

### 9.8.1 学习率调度

In [ ]:
# 学习率调度的建议
print("学习率调度建议:")
print("=" * 50)
print("\n1. 初始学习率选择:")
print("   - 从较小的值开始 (0.001 或 0.0001)")
print("   - 如果 loss 不下降，尝试增大")
print("   - 如果 loss 震荡，尝试减小")

print("\n2. 学习率衰减:")
print("   - StepLR: 每隔 N 个 epoch 衰减")
print("   - ReduceLROnPlateau: 当指标不再提升时衰减")
print("   - CosineAnnealing: 余弦退火")

print("\n3. 预训练和微调:")
print("   - 使用较大学习率预训练")
print("   - 使用较小学习率微调")

### 9.8.2 正则化技巧

In [ ]:
# 正则化技巧
print("正则化技巧:")
print("=" * 50)

regularization_tips = {
    "Dropout": "随机丢弃神经元，防止过拟合 (推荐 0.1-0.5)",
    "Weight Decay": "L2 正则化，限制权重大小",
    "Batch Normalization": "批归一化，加速训练",
    "Layer Normalization": "层归一化，适用于 RNN",
    "Early Stopping": "早停，防止过拟合",
    "Data Augmentation": "数据增强，增加数据多样性",
}

for technique, desc in regularization_tips.items():
    print(f"\n{technique}:")
    print(f"  {desc}")

## 9.9 实践练习

In [ ]:
# 练习1: 调整 LSTM 的超参数
# 尝试不同的 hidden_size 和 num_layers
# 对比效果

# 你的代码



# 参考答案
# model_configs = [
#     {'hidden_size': 32, 'num_layers': 1},
#     {'hidden_size': 64, 'num_layers': 2},
#     {'hidden_size': 128, 'num_layers': 2},
# ]
# for config in model_configs:
#     model = LSTM(d_feat=6, **config, n_epochs=20)
#     model.fit(dataset)
#     pred = model.predict(dataset)
#     metrics = evaluate_predictions(pred, labels)
#     print(f"{config}: IC={metrics['IC']:.4f}")

In [ ]:
# 练习2: 添加 Dropout 正则化
# 尝试不同的 dropout 值

# 你的代码



# 提示：修改 dropout 参数

In [ ]:
# 练习3: 实现模型融合
# 将 LSTM 和 GRU 的预测结果取平均
# 评估融合后的效果

# 你的代码



# 参考答案
# ensemble_pred = (lstm_predictions + gru_predictions) / 2
# ensemble_metrics = evaluate_predictions(ensemble_pred, labels)
# print(f"融合模型 IC: {ensemble_metrics['IC']:.4f}")

## 9.10 本章小结

本章我们学习了：

1. **深度学习模型类型**：
   - LSTM: 长短期记忆网络
   - GRU: 门控循环单元
   - Transformer: 自注意力机制

2. **模型配置与训练**：
   - 使用 TSDatasetH 处理时序数据
   - 配置模型超参数
   - 训练与评估

3. **模型对比**：
   - IC 和 Rank IC 指标
   - 可视化对比

4. **训练技巧**：
   - 学习率调度
   - 正则化方法

### 关键参数说明

| 参数 | 说明 | 典型范围 |
|------|------|----------|
| `d_feat` | 输入特征维度 | 6 (Alpha360) |
| `hidden_size` | 隐藏层大小 | 32 - 256 |
| `num_layers` | 网络层数 | 1 - 4 |
| `dropout` | Dropout 比例 | 0.0 - 0.5 |
| `n_epochs` | 训练轮数 | 50 - 200 |
| `lr` | 学习率 | 0.0001 - 0.01 |
| `batch_size` | 批大小 | 200 - 2000 |

### 下一章预告

下一章我们将学习集成学习与模型融合，包括：
- 集成学习方法
- 模型融合策略
- 特征重要性分析